In [1]:
pip install transformers torch sentencepiece

  Using cached transformers-4.54.0-py3-none-any.whl.metadata (41 kB)
  Using cached torch-2.7.1-cp312-none-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached huggingface_hub-0.34.1-py3-none-any.whl.metadata (14 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.1 kB)
  Using cached regex-2024.11.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.21.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 

In [1]:
import os
import json
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def transliterate_text(text, transliteration_pipeline):
    """
    Transliterates a given text using the Hugging Face pipeline.
    """
    if not text:
        return ""
    try:
        # The exact key for the output might vary by model.
        # For 'kasunw/sinhala-transliterator', it typically gives a 'translation_text' or similar.
        # We will assume 'generated_text' as a common key for text2text-generation,
        # but if it fails, you might need to inspect the pipeline output structure.
        result = transliteration_pipeline(text)
        return result[0]['generated_text'] # This is a common key, verify if needed
    except Exception as e:
        print(f"Error during transliteration: {e}")
        return text # Return original text on error

In [ ]:
def process_metadata_files(root_folder, transliteration_model_name):

    print(f"Loading transliteration model: {transliteration_model_name}...")
    try:
        # 'kasunw/sinhala-transliterator' uses a Sequence-to-Sequence model
        tokenizer = AutoTokenizer.from_pretrained(transliteration_model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(transliteration_model_name)
        transliteration_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer)
        print("Model loaded successfully.")
    except Exception as e:
        print(f"Error loading model: {e}. Please check the model name and your internet connection.")
        return

    for dirpath, dirnames, filenames in os.walk(root_folder):
        if "metadata.json" in filenames:
            json_file_path = os.path.join(dirpath, "metadata.json")
            print(f"Processing: {json_file_path}")

            try:
                with open(json_file_path, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)

                # Transliterate 'title' if it exists and 'title_en' doesn't
                if 'title' in metadata:
                    original_title = metadata['title']
                    transliterated_title = transliterate_text(original_title, transliteration_pipeline)
                    metadata['title_en'] = transliterated_title
                    print(f"  Title: '{original_title}' -> Title_en: '{transliterated_title}'")
                elif 'title_en' in metadata:
                    print(f"  'title_en' already exists for this file. Skipping title transliteration.")
                else:
                    print(f"  'title' key not found in metadata for {json_file_path}. Skipping title transliteration.")

                # Transliterate 'author' if it exists and 'author_en' doesn't
                if 'author' in metadata and 'author_en' not in metadata:
                    original_author = metadata['author']
                    transliterated_author = transliterate_text(original_author, transliteration_pipeline)
                    metadata['author_en'] = transliterated_author
                    print(f"  Author: '{original_author}' -> Author_en: '{transliterated_author}'")
                elif 'author_en' in metadata:
                    print(f"  'author_en' already exists for this file. Skipping author transliteration.")
                else:
                    print(f"  'author' key not found in metadata for {json_file_path}. Skipping author transliteration.")

                # Write the updated metadata back to the file
                with open(json_file_path, 'w', encoding='utf-8') as f:
                    json.dump(metadata, f, indent=4, ensure_ascii=False) # indent for readability, ensure_ascii=False for non-ASCII characters
                print(f"Updated {json_file_path}\n")

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON in {json_file_path}: {e}")
            except Exception as e:
                print(f"An unexpected error occurred with {json_file_path}: {e}")


In [4]:
if __name__ == "__main__":
    # --- Configuration ---
    # As per saved information, your data folder is named 'data'.
    root_folder_path = "data/OCR_Final"

    # Using the specified transliteration model
    transliteration_model_name = "kasunw/sinhala-transliterator"

    # --- Run the process ---
    process_metadata_files(root_folder_path, transliteration_model_name)
    print("Metadata file processing complete.")

Loading transliteration model: kasunw/sinhala-transliterator...


Device set to use mps:0


Model loaded successfully.
Processing: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json
  Title: 'ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව' -> Title_en: 'තරම් ප්රතිව්ධපිකයා හැදුවා මහාභූජනයන්ගෙ පරක්කුතාවය'
  Author: 'Unknown' -> Author_en: 'Unknown'
Updated data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json

Processing: data/OCR_Final/නිදහසේ මන්ත්‍රය/metadata.json
  Title: 'නිදහසේ මන්ත්‍රය' -> Title_en: 'නිදහසේ මුණගැතියි'
  Author: 'ඇස් මහින්ද හිමි' -> Author_en: 'ඇන්සි බොරුට එන්න'
Updated data/OCR_Final/නිදහසේ මන්ත්‍රය/metadata.json

Processing: data/OCR_Final/පැරණි ගම/metadata.json
  Title: 'පැරණි ගම' -> Title_en: 'පරණිි ගමම'
  Author: 'ගල්පාත ඛේමානන්ද හිමි' -> Author_en: 'ගාලපාත ඛේදනයන්නද හෙන'
Updated data/OCR_Final/පැරණි ගම/metadata.json

Processing: data/OCR_Final/පන්සිය පනස් ජාතක පොත/metadata.json
  Title: 'පන්සිය පනස් ජාතක පොත' -> Title_en: 'පින්සිය පින්සෙස් ජනතාවට පොහොත'
  Author: 'Unknown' -> Author_en: 'Unknown'
Updated data/O

KeyboardInterrupt: 